# SQL 생성 품질 테스트
- `USE_DB = True`: 사내망 연결 시 DB에서 실시간 조회
- `USE_DB = False`: 하드코딩 샘플 데이터로 오프라인 테스트

In [ ]:
import os, json
from pathlib import Path
import anthropic

# ── 설정 ──────────────────────────────────────────────
USE_DB = False  # True: 사내망 DB 조회 / False: 샘플 데이터
LIMIT  = 5      # DB 조회 시 건수

# .env 로드
env_path = Path(".") / ".env"
for line in env_path.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        k, _, v = line.partition("=")
        os.environ.setdefault(k.strip(), v.strip())

PROMPTS_DIR = Path(".") / "dags" / "prompts"
print("설정 완료")

In [ ]:
# ── 샘플 데이터 (오프라인 테스트용) ───────────────────
SAMPLE_DOCS = [
    # 실제 데이터는 로컬에서만 사용 (git 미포함)
    # {
    #     "doc_id": 0,
    #     "user_nm": "홍길동",
    #     "dept_nm": "팀명",
    #     "doc_title": "결재 제목",
    #     "doc_contents": "결재 내용",
    # },
]

if USE_DB:
    import pymysql
    conn = pymysql.connect(
        host=os.environ["GW_DB_HOST"],
        port=int(os.environ.get("GW_DB_PORT", 13306)),
        user=os.environ["GW_DB_USER"],
        password=os.environ["GW_DB_PASSWORD"],
        database=os.environ["GW_DB_NAME"],
        charset="utf8mb4",
    )
    with conn.cursor(pymysql.cursors.DictCursor) as cur:
        cur.execute("""
            SELECT ad.doc_id, ad.user_nm, ad.dept_nm, ad.doc_title, adcw.doc_contents
            FROM neos.teag_appdoc ad
            INNER JOIN neos.teag_appdoc_contents adc ON ad.doc_id = adc.doc_id
            INNER JOIN neos.teag_appdoc_contents_word adcw ON adc.doc_id = adcw.doc_id
            WHERE ad.form_id = 144 AND ad.doc_sts IN (30, 90)
            ORDER BY ad.doc_id DESC LIMIT %s
        """, (LIMIT,))
        docs = cur.fetchall()
    conn.close()
    print(f"DB 조회: {len(docs)}건")
else:
    docs = SAMPLE_DOCS
    print(f"샘플 데이터 사용: {len(docs)}건")

for d in docs:
    print(f"  [{d['doc_id']}] {d['doc_title']}")

In [ ]:
# Claude 클라이언트 + 프롬프트 로드
client = anthropic.Anthropic(
    base_url=os.environ.get("ANTHROPIC_BASE_URL"),
    api_key=os.environ["ANTHROPIC_API_KEY"],
)
schema = (PROMPTS_DIR / "schema.md").read_text(encoding="utf-8")
fewshot = (PROMPTS_DIR / "fewshot_all.md").read_text(encoding="utf-8")

system_content = (
    "당신은 밀리의서재 개인정보 추출 BigQuery SQL 전문가입니다.\n\n"
    "## 규칙\n"
    "- mem_seq(회원번호)만 추출 — 개인정보 직접 추출 금지\n"
    "- 테이블은 millie-analysis GCP 프로젝트만 사용\n"
    "- 공통 필터 항상 포함: test_yn='N', millie_yn='N', dormant_yn='N', member_status != '탈퇴회원'\n"
    "- 조건 불명확 시 SQL 대신 확인 필요 사항을 명시\n"
    "- BigQuery 표준 SQL 문법 사용 (백틱으로 테이블 경로 감싸기)\n\n"
    f"## 테이블 스키마\n{schema}\n\n"
    f"## 과거 처리 예시\n{fewshot}"
)
print("프롬프트 로드 완료")

In [ ]:
def parse_json(raw):
    try:
        if "```" in raw:
            s = raw.split("```json")[-1].split("```")[0].strip()
        else:
            s = raw[raw.index("{"):raw.rindex("}")+1]
        return json.loads(s)
    except Exception:
        return {"parse_error": True, "raw": raw}

results = []

for doc in docs:
    print(f"\n{'='*60}")
    print(f"[{doc['doc_id']}] {doc['doc_title']}")
    print(f"요청자: {doc['user_nm']} ({doc['dept_nm']})")

    # 1차: 유형 분류
    cls_msg = client.messages.create(
        model="claude-4.5-haiku",
        max_tokens=200,
        messages=[{"role": "user", "content": (
            f"아래 전자결재의 추출 요청 유형을 분류하세요.\n\n"
            f"유형: 이벤트_참여자 / 알림_신청자 / 마케팅_구독조건 / 기타_특수\n"
            f"결재 제목: {doc['doc_title']}\n"
            f"결재 내용: {doc['doc_contents']}\n\n"
            f'JSON으로만 응답: {{"doc_id": {doc["doc_id"]}, "type": "유형", "reason": "근거"}}'
        )}],
    )
    classified = parse_json(cls_msg.content[0].text.strip())
    doc_type = classified.get("type", "기타_특수")
    print(f"유형: {doc_type} — {classified.get('reason')}")

    # 2차: SQL 생성
    sql_msg = client.messages.create(
        model="claude-4.5-sonnet",
        max_tokens=2000,
        system=[{"type": "text", "text": system_content, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": (
            f"결재 유형: {doc_type}\n"
            f"결재 제목: {doc['doc_title']}\n"
            f"결재 내용:\n{doc['doc_contents']}\n\n"
            f'반드시 JSON으로만 응답:\n'
            f'{{"doc_id": {doc["doc_id"]}, "request_type": "{doc_type}", '
            f'"conditions": "핵심 조건 한 줄 요약", '
            f'"sql_draft": "SQL 또는 확인필요내용", '
            f'"needs_review": true또는false}}'
        )}],
    )
    result = parse_json(sql_msg.content[0].text.strip())

    if result.get("parse_error"):
        print(f"파싱 실패: {result.get('raw', '')[:200]}")
    else:
        print(f"조건: {result.get('conditions')}")
        print(f"검토 필요: {result.get('needs_review')}")
        print(f"\nSQL 초안:\n{result.get('sql_draft')}")

    results.append({"doc_id": doc["doc_id"], "title": doc["doc_title"], "type": doc_type, "result": result})